# nnU-Net runs for the Angus claw segmentation paper

This notebook only **trains** nnU-Net. Evaluation on the test hooves is done locally, so no test image is uploaded.

It answers the open points about the nnU-Net baseline:

1. **Three independent runs.** The split file contains the same split three times (folds 0, 1, 2), so each fold is an
   independent training run on identical data. These give mean ± SD, as for the three U-Net seeds.
2. **Training length.** Optional runs with 250 and with the full default 1000 epochs, to show whether 100 epochs were
   enough.

The configuration must equal the local one (nnunetv2 2.7.0, 2D, patch 320 x 320, batch size 8); cell 5 checks this.
Results are written to Google Drive, so an interrupted run continues from its last checkpoint when the notebook is run again.

**Data.** The package contains training and validation hoof windows owned by Te Mania Angus. Do not share it further,
and delete the Drive folder after the results have been downloaded.

Expected time per 100-epoch run on a large GPU: about 20 to 40 minutes. The 1000-epoch run takes several hours.

In [ ]:
# 1. GPU
!nvidia-smi

In [ ]:
# 2. Google Drive (results persist across disconnects)
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/angusclaw_nnunet'
os.makedirs(WORK, exist_ok=True)
print(WORK)

In [ ]:
# 3. nnU-Net, same version as the local runs
!pip install -q nnunetv2==2.7.0
import nnunetv2, torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 4. Data package: put angusclaw_nnunet_package.zip into the Drive folder above, or upload it when asked
import shutil, zipfile
PKG = f'{WORK}/angusclaw_nnunet_package.zip'
if not os.path.exists(PKG):
    from google.colab import files
    uploaded = files.upload()          # choose angusclaw_nnunet_package.zip
    shutil.move(next(iter(uploaded)), PKG)
os.makedirs('/content/nnunet/raw', exist_ok=True)
with zipfile.ZipFile(PKG) as z:
    z.extractall('/content/nnunet/pkg')
if not os.path.exists('/content/nnunet/raw/Dataset501_AngusClaw'):
    shutil.copytree('/content/nnunet/pkg/Dataset501_AngusClaw', '/content/nnunet/raw/Dataset501_AngusClaw')
os.environ['nnUNet_raw'] = '/content/nnunet/raw'
os.environ['nnUNet_preprocessed'] = '/content/nnunet/preprocessed'
os.environ['nnUNet_results'] = f'{WORK}/results'
print(len(os.listdir('/content/nnunet/raw/Dataset501_AngusClaw/imagesTr')), 'training + validation images')

In [ ]:
# 5. Planning and preprocessing, then check that the configuration equals the local one
!nnUNetv2_plan_and_preprocess -d 501 -c 2d --verify_dataset_integrity -np 8
import json
new = json.load(open('/content/nnunet/preprocessed/Dataset501_AngusClaw/nnUNetPlans.json'))['configurations']['2d']
ref = json.load(open('/content/nnunet/pkg/reference_nnUNetPlans.json'))['configurations']['2d']
for key in ('patch_size', 'batch_size', 'normalization_schemes'):
    print(key, new[key], '| local:', ref[key])
    assert new[key] == ref[key], f'{key} differs from the local configuration'
assert new['architecture'] == ref['architecture'], 'network architecture differs from the local configuration'
shutil.copy('/content/nnunet/pkg/splits_final_3runs.json', '/content/nnunet/preprocessed/Dataset501_AngusClaw/splits_final.json')
print('configuration identical to the local run; split file installed (3 identical folds)')

In [ ]:
# 6. Training runs. Comment out the optional ones if time is short.
RUNS = [
    ('nnUNetTrainer_100epochs', 0),
    ('nnUNetTrainer_100epochs', 1),
    ('nnUNetTrainer_100epochs', 2),
    ('nnUNetTrainer_250epochs', 0),   # optional: is 100 epochs enough?
    ('nnUNetTrainer', 0),             # optional: full default schedule, 1000 epochs (several hours)
]
import subprocess, time
for trainer, fold in RUNS:
    out = f"{os.environ['nnUNet_results']}/Dataset501_AngusClaw/{trainer}__nnUNetPlans__2d/fold_{fold}"
    if os.path.exists(f'{out}/checkpoint_final.pth'):
        print('done already:', trainer, fold); continue
    cmd = ['nnUNetv2_train', '501', '2d', str(fold), '-tr', trainer]
    if os.path.exists(f'{out}/checkpoint_latest.pth'):
        cmd.append('--c')                 # continue an interrupted run
    t0 = time.time()
    print(' '.join(cmd), flush=True)
    # stream the nnU-Net log into the cell (subprocess.run would not show it in Colab)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'{trainer} fold {fold} failed')
    print(f'{trainer} fold {fold}: {(time.time() - t0) / 60:.1f} min', flush=True)

In [ ]:
# 7. Package the trained models and logs (no images) and download them
import glob
res = f"{os.environ['nnUNet_results']}/Dataset501_AngusClaw"
zip_path = f'{WORK}/angusclaw_nnunet_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob(f'{res}/*/*.json') + glob.glob(f'{res}/*/fold_*/*'):
        name = os.path.basename(path)
        if name in ('checkpoint_final.pth', 'progress.png', 'debug.json') or name.startswith('training_log') or path.endswith('.json'):
            z.write(path, os.path.relpath(path, res))
print(zip_path, round(os.path.getsize(zip_path) / 1e6), 'MB')
from google.colab import files
files.download(zip_path)

## Back on the local machine

Unzip `angusclaw_nnunet_results.zip` into `nnunet/results_colab/Dataset501_AngusClaw/` (a separate folder, so the
local fold-0 run is not overwritten). The local benchmark then evaluates every run on the test hooves.

Afterwards delete `angusclaw_nnunet_package.zip` and the `angusclaw_nnunet` folder from Google Drive.